# Kaggle GPU — Fine-tune VietOCR (CRAFT detector + UPSCALING)

Full-training version of the upscale experiment. Same proven CRAFT pipeline + upscale
small thumbnails. Outputs `ocr_vietocr_ftup_{test,all}.parquet` to A/B against the
current `vietocr_ft` (no upscale) on cross-validation.

## Setup
1. New Notebook inside the competition, **GPU T4**, **Internet ON**.
2. Run **Cell 1** -> **Restart & clear cell outputs** (Pillow pin) -> Run All.
3. **For the full run** keep `MAX_TRAIN_IMAGES=None` (Cell 3), `ITERS=15000` (Cell 4) and use
   **Save Version > Save & Run All (Commit)** (server-side, no interactive freeze).
4. Download `ocr_vietocr_ftup_test.parquet` + `_all` into local `cache/`.

In [ ]:
# Install. Pillow pin AFTER (else ImageFont/_util.is_directory). RESTART kernel after this cell.
!pip install -q easyocr vietocr rapidfuzz
!pip install -q --force-reinstall 'Pillow==10.4.0'
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
import os, re, zipfile, unicodedata, random
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm.auto import tqdm

KIN = Path('/kaggle/input'); ROOT = None
for p in KIN.rglob('test.csv'):
    ROOT = p.parent; break
assert ROOT is not None, 'competition data not mounted'
WORK = Path('/kaggle/working'); print('ROOT =', ROOT)

def _imgs(kind):                      # kind = 'train' or 'test'
    want_train = (kind == 'train')
    for d in KIN.rglob('*'):
        if d.is_dir() and any(d.glob('img_*.jpg')):
            if ('train' in str(d).lower()) == want_train:
                return d
    for z in KIN.rglob('*.zip'):
        if ('train' in z.name.lower()) != want_train:
            continue
        ex = WORK / (kind + '_imgs'); ex.mkdir(parents=True, exist_ok=True)
        if not any(ex.rglob('img_*.jpg')):
            with zipfile.ZipFile(z) as zf: zf.extractall(ex)
        for d in [ex, *[c for c in ex.rglob('*') if c.is_dir()]]:
            if any(d.glob('img_*.jpg')): return d
    print('--- /kaggle/input tree ---')
    for p in sorted(KIN.rglob('*')):
        if p.is_dir() or p.suffix.lower() in ('.zip', '.csv'): print(p)
    raise FileNotFoundError(kind + ' images not found')

TRAIN_DIR = _imgs('train'); TEST_DIR = _imgs('test')
labels = pd.read_csv(ROOT/'train_labels.csv', keep_default_na=False)
labels['ocr_text'] = labels['ocr_text'].astype(str).str.strip()
test_ids = pd.read_csv(ROOT/'test.csv')['image_id'].tolist()
print('train', len(list(TRAIN_DIR.glob('*.jpg'))), '| test', len(list(TEST_DIR.glob('*.jpg'))), '| labeled', len(labels))

def fold(s):
    s = unicodedata.normalize('NFD', str(s).lower())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = s.replace(chr(0x111), 'd'); s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
def clean_ocr(t, maxlen=500):
    t = unicodedata.normalize('NFC', str(t)).replace(chr(10),' ').replace(chr(9),' ')
    t = re.sub(r'\s+', ' ', t).strip(); toks = t.split(); ded = [toks[0]] if toks else []
    for w in toks[1:]:
        if w.lower() != ded[-1].lower(): ded.append(w)
    t = ' '.join(ded); return t[:maxlen].rstrip() if len(t) > maxlen else t
def load_upscaled(path, min_side=720):
    img = cv2.imread(str(path))
    if img is None: return None
    h, w = img.shape[:2]
    if min(h, w) < min_side:
        s = min_side / min(h, w); img = cv2.resize(img, (int(w*s), int(h*s)), interpolation=cv2.INTER_CUBIC)
    return img

In [ ]:
# ---- CRAFT detect (on upscaled img) + align crops to GT spans ----
import easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from rapidfuzz import fuzz

MAX_TRAIN_IMAGES = None   # 1500 for a quick run; None = full
MATCH_MIN = 78

detector = easyocr.Reader(['vi','en'], gpu=True, verbose=False)
def detect_boxes(img):
    try:
        horiz,_ = detector.detect(img, min_size=15, text_threshold=0.6)
        return [tuple(int(v) for v in b) for b in (horiz[0] if horiz else [])]  # (x0,x1,y0,y1)
    except Exception:
        return []

base_cfg = Cfg.load_config_from_name('vgg_transformer')
base_cfg['device']='cuda:0'; base_cfg['predictor']['beamsearch']=False
base_rec = Predictor(base_cfg); VOCAB = set(base_cfg['vocab'])
LINES = WORK/'lines'; LINES.mkdir(exist_ok=True)

def best_gt_span(pred, gt_tokens, max_span=10):
    pf=fold(pred)
    if len(pf)<2: return None,0
    best,bs,n=None,0,len(gt_tokens)
    for i in range(n):
        for L in range(1,max_span+1):
            if i+L>n: break
            span=' '.join(gt_tokens[i:i+L]); sc=fuzz.ratio(pf,fold(span))
            if sc>bs: bs,best=sc,span
        if bs==100: break
    return best,bs

rows = labels[labels.ocr_text!=''].copy()
if MAX_TRAIN_IMAGES: rows = rows.sample(min(MAX_TRAIN_IMAGES,len(rows)), random_state=42)
pairs=[]; idx=0
for _,r in tqdm(rows.iterrows(), total=len(rows), desc='align'):
    img = load_upscaled(TRAIN_DIR/(r.image_id+'.jpg'))
    if img is None: continue
    gt_tokens=r.ocr_text.split()
    for (x0,x1,y0,y1) in detect_boxes(img):
        x0,y0=max(0,x0),max(0,y0); x1,y1=min(img.shape[1],x1),min(img.shape[0],y1)
        if x1-x0<6 or y1-y0<6: continue
        crop=img[y0:y1,x0:x1]
        try: pred=base_rec.predict(Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)))
        except Exception: continue
        span,sc=best_gt_span(pred,gt_tokens)
        if span is None or sc<MATCH_MIN: continue
        lb=''.join(ch for ch in span if ch in VOCAB).strip()
        if len(lb)<1: continue
        fn='l%06d.png'%idx; cv2.imwrite(str(LINES/fn),crop); idx+=1; pairs.append((fn,lb))
print('\nbuilt',len(pairs),'pairs'); random.seed(0); random.shuffle(pairs)
with open(LINES/'train_ann.txt','w',encoding='utf-8') as f:
    for fn,lb in pairs: f.write(fn+'\t'+lb+'\n')
print('SAMPLES (eyeball clean Vietnamese):'); [print('  ',lb) for _,lb in pairs[:12]]

In [ ]:
# ---- Fine-tune (numpy/lmdb/val fixes) ----
import numpy as np, time
if not hasattr(np,'sctypes'):
    np.sctypes={'int':[np.int8,np.int16,np.int32,np.int64],'uint':[np.uint8,np.uint16,np.uint32,np.uint64],'float':[np.float16,np.float32,np.float64],'complex':[np.complex64,np.complex128],'others':[bool,object,bytes,str,np.void]}
for _n,_t in [('bool',bool),('float',float),('int',int),('object',object),('str',str),('complex',complex)]:
    if not hasattr(np,_n): setattr(np,_n,_t)
_f=np.fromstring; np.fromstring=lambda s,dtype=float,count=-1,sep='': (np.frombuffer(s,dtype=dtype,count=count) if sep=='' else _f(s,dtype=dtype,count=count,sep=sep))
from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg
ITERS=15000   # 3000 for a quick run
cfg=Cfg.load_config_from_name('vgg_transformer'); cfg['device']='cuda:0'; cfg['dataloader']={'num_workers':2}
cfg['dataset']={'name':'uraup_%d'%int(time.time()),'data_root':str(LINES),'train_annotation':'train_ann.txt','valid_annotation':None,'image_height':32,'image_min_width':32,'image_max_width':512}
cfg['trainer']={'batch_size':32,'print_every':200,'valid_every':10**9,'iters':ITERS,'export':'./vietocr_ftup.pth','checkpoint':'./ckptup.pth','log':'./up.log','metrics':5000}
tr=Trainer(cfg,pretrained=True); tr.train(); tr.save_weights('./vietocr_ftup.pth'); print('saved ./vietocr_ftup.pth')

In [ ]:
# ---- Inference (CRAFT + upscale + fine-tuned) -> parquet ----
ft_cfg=Cfg.load_config_from_name('vgg_transformer'); ft_cfg['device']='cuda:0'; ft_cfg['predictor']['beamsearch']=False; ft_cfg['weights']='./vietocr_ftup.pth'
ft_rec=Predictor(ft_cfg)
def ocr_image(path):
    img=load_upscaled(path)
    if img is None: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    items=[]
    for (x0,x1,y0,y1) in detect_boxes(img):
        x0,y0=max(0,x0),max(0,y0); x1,y1=min(img.shape[1],x1),min(img.shape[0],y1)
        if x1-x0<6 or y1-y0<6: continue
        items.append((y0,x0,Image.fromarray(cv2.cvtColor(img[y0:y1,x0:x1],cv2.COLOR_BGR2RGB))))
    if not items: return {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}
    try: texts=ft_rec.predict_batch([it[2] for it in items])
    except Exception: texts=[ft_rec.predict(it[2]) for it in items]
    ys=[it[0] for it in items]; band=max(8.0,(max(ys)-min(ys))/40.0)
    order=sorted(range(len(items)), key=lambda i:(round(items[i][0]/band),items[i][1]))
    text=clean_ocr(' '.join(texts[i] for i in order if str(texts[i]).strip()))
    return {'ocr_text':text,'raw_text':text,'mean_conf':0.0,'n_boxes':len(items),'n_chars':len(text)}
def run(ids,img_dir,out,save_every=200):
    done={}
    if Path(out).exists(): done={r['image_id']:r for r in pd.read_parquet(out).to_dict('records')}
    recs=list(done.values()); pend=[i for i in ids if i not in done]; print(out,'pending',len(pend),'/',len(ids))
    for k,iid in enumerate(tqdm(pend)):
        r=ocr_image(img_dir/(iid+'.jpg')); r['image_id']=iid; recs.append(r)
        if (k+1)%save_every==0: pd.DataFrame(recs).to_parquet(out,index=False)
    pd.DataFrame(recs).to_parquet(out,index=False); print('saved',out,'fill',(pd.DataFrame(recs).ocr_text.str.strip()!='').mean())
run(test_ids, TEST_DIR, '/kaggle/working/ocr_vietocr_ftup_test.parquet')
run(labels['image_id'].tolist(), TRAIN_DIR, '/kaggle/working/ocr_vietocr_ftup_all.parquet')
print('\nDONE — download ocr_vietocr_ftup_test.parquet (+ _all).')